# Task 1: Symbolic, Unconditioned MIDI Generation

**Group:** Charles N, Dennis C

This notebook trains symbolic music generators on the MAESTRO v3.0.0 MIDI-only dataset and exports a generated acoustic-guitar MIDI file named `symbolic_unconditioned.mid`.

The task is formulated as unconditioned autoregressive sequence modeling: learn a music distribution `p(x)` from symbolic note tokens, then sample new symbolic music without any external prompt. The notebook includes a random baseline, an interpretable trigram Markov baseline, and a small LSTM language model. A compact Transformer decoder architecture is also included for comparison and discussion, but the LSTM is the neural model trained here to keep runtime manageable.

## 1. Setup and Dependencies

Run the dependency cell once if your environment does not already have the required packages. The notebook uses the MIDI-only MAESTRO dataset, not the audio files.

If your Python install reports an `externally-managed-environment` error, launch Jupyter from a virtual environment and then run this notebook.

In [1]:
# Install dependencies inside the active notebook kernel if needed.
# If these packages are already installed, this cell should finish quickly.
# If this fails with an externally-managed-environment error, create and use a venv first:
#   python3 -m venv .venv
#   source .venv/bin/activate
#   python -m pip install jupyter pretty_midi pandas matplotlib scipy torch
%pip install -q pretty_midi pandas matplotlib scipy torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import random
import shutil
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pretty_midi

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc

# Reproducibility
RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(RANDOM_SEED)

# Works when the notebook is launched either from the repo root or from Task1/.
DATASET_CANDIDATES = [
    Path("maestro-v3.0.0"),
    Path("Task1") / "maestro-v3.0.0",
]
DATASET_DIR = next((path for path in DATASET_CANDIDATES if path.exists()), DATASET_CANDIDATES[0])
OUTPUT_DIR = DATASET_DIR.parent

# Keep the subset modest so the notebook and exported HTML are reproducible on normal laptops.
MAX_FILES = 200

# Timing normalization and tokenization settings.
TIME_STEP_SECONDS = 0.125
MAX_SILENCE_SECONDS = 2.0
MIN_DURATION_SECONDS = 0.05
MAX_DURATION_SECONDS = 4.0
DURATION_BUCKETS = np.array([0.125, 0.25, 0.5, 1.0, 2.0, 4.0], dtype=float)

# Markov and generation settings.
N_GRAM = 3
GENERATED_NOTE_COUNT = 300
TEMPO_BPM = 120
GENERATED_VELOCITY = 80

# Neural model settings. These are intentionally small for a course notebook.
CONTEXT_LENGTH = 64
BATCH_SIZE = 64
LSTM_EPOCHS = 3
MAX_TRAIN_WINDOWS = 5000
MAX_VAL_WINDOWS = 1000
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
TOP_K = 40
TEMPERATURE = 1.20

GM_PROGRAMS = {
    "piano": 0,
    "acoustic_guitar_nylon": 24,
    "acoustic_guitar_steel": 25,
    "electric_guitar_clean": 27,
}

FINAL_OUTPUT_PATH = OUTPUT_DIR / "symbolic_unconditioned.mid"
MARKOV_OUTPUT_PATH = OUTPUT_DIR / "markov_acoustic_guitar.mid"
BASELINE_OUTPUT_PATH = OUTPUT_DIR / "random_baseline.mid"
LSTM_PIANO_OUTPUT_PATH = OUTPUT_DIR / "lstm_piano.mid"
LSTM_ACOUSTIC_GUITAR_OUTPUT_PATH = OUTPUT_DIR / "lstm_acoustic_guitar.mid"
LSTM_ELECTRIC_GUITAR_OUTPUT_PATH = OUTPUT_DIR / "lstm_electric_guitar.mid"

print(f"Dataset directory: {DATASET_DIR.resolve()}")
print(f"Output directory:  {OUTPUT_DIR.resolve()}")
print(f"PyTorch available: {TORCH_AVAILABLE}")

Dataset directory: /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1/maestro-v3.0.0
Output directory:  /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1
PyTorch available: True


## 2. Preprocessing

The dataset is MAESTRO v3.0.0 MIDI-only data from Google Magenta: <https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip>. MAESTRO contains classical piano performances captured on Yamaha Disklavier instruments, producing high-quality MIDI with expressive timing and velocity.

The preprocessing stage recursively finds MIDI files, removes corrupted/unparseable files by catching parser failures, extracts note-level information, normalizes timing to a fixed resolution, clips long silences, filters extremely short and extremely long notes, and constructs a token vocabulary. Each training token is `(pitch, duration_bucket)`. Start/end times, raw durations, clipped inter-onset gaps, and velocities are retained for analysis.

In [3]:
def find_midi_files(dataset_dir):
    """Recursively find .mid and .midi files."""
    midi_files = list(dataset_dir.rglob("*.mid")) + list(dataset_dir.rglob("*.midi"))
    return sorted(midi_files)


def is_piano_instrument(instrument):
    """General MIDI programs 0-7 are piano-family instruments."""
    return (not instrument.is_drum) and (0 <= instrument.program <= 7)


def quantize_duration(duration_seconds, buckets=DURATION_BUCKETS):
    """Map a duration in seconds to the closest predefined bucket."""
    bucket_index = int(np.argmin(np.abs(buckets - duration_seconds)))
    return float(buckets[bucket_index])


def normalize_time(value_seconds, step_seconds=TIME_STEP_SECONDS):
    """Quantize a time value to an integer grid step."""
    return int(round(float(value_seconds) / step_seconds))


def extract_tokens_from_midi(midi_path):
    """Return note tokens and note metadata for one MIDI file."""
    midi = pretty_midi.PrettyMIDI(str(midi_path))

    piano_instruments = [inst for inst in midi.instruments if is_piano_instrument(inst)]
    selected_instruments = piano_instruments
    if not selected_instruments:
        selected_instruments = [inst for inst in midi.instruments if not inst.is_drum]

    note_rows = []
    for instrument in selected_instruments:
        for note in instrument.notes:
            raw_duration = float(note.end - note.start)
            if raw_duration < MIN_DURATION_SECONDS or raw_duration > MAX_DURATION_SECONDS:
                continue

            duration_bucket = quantize_duration(raw_duration)
            note_rows.append(
                {
                    "file": str(midi_path),
                    "start": float(note.start),
                    "end": float(note.end),
                    "pitch": int(note.pitch),
                    "raw_duration": raw_duration,
                    "duration_bucket": duration_bucket,
                    "velocity": int(note.velocity),
                    "program": int(instrument.program),
                    "instrument_name": instrument.name,
                }
            )

    note_rows.sort(key=lambda row: (row["start"], row["pitch"], row["duration_bucket"]))

    previous_start = None
    for row in note_rows:
        if previous_start is None:
            gap = 0.0
        else:
            gap = max(0.0, row["start"] - previous_start)
        row["gap_since_previous"] = gap
        row["clipped_gap"] = min(gap, MAX_SILENCE_SECONDS)
        row["start_step"] = normalize_time(row["start"])
        row["end_step"] = normalize_time(row["end"])
        row["duration_steps"] = max(1, normalize_time(row["raw_duration"]))
        row["clipped_gap_steps"] = normalize_time(row["clipped_gap"])
        previous_start = row["start"]

    tokens = [(row["pitch"], row["duration_bucket"]) for row in note_rows]
    return tokens, note_rows


midi_files = find_midi_files(DATASET_DIR)
selected_midi_files = midi_files[:MAX_FILES]

print(f"MIDI files found:     {len(midi_files)}")
print(f"MIDI files selected:  {len(selected_midi_files)}")
print("First selected file:", selected_midi_files[0] if selected_midi_files else "None")

MIDI files found:     1276
MIDI files selected:  200
First selected file: maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi


In [4]:
sequences = []
file_summaries = []
all_note_rows = []
failures = []

for file_index, midi_path in enumerate(selected_midi_files, start=1):
    try:
        tokens, note_rows = extract_tokens_from_midi(midi_path)
        if len(tokens) >= N_GRAM:
            sequences.append(tokens)
            all_note_rows.extend(note_rows)
            file_summaries.append(
                {
                    "file": str(midi_path),
                    "note_tokens": len(tokens),
                    "first_pitch": tokens[0][0],
                    "last_pitch": tokens[-1][0],
                }
            )
    except Exception as exc:
        failures.append({"file": str(midi_path), "error": repr(exc)})

    if file_index % 25 == 0 or file_index == len(selected_midi_files):
        print(f"Processed {file_index:>3}/{len(selected_midi_files)} files")

file_summary_df = pd.DataFrame(file_summaries)
note_df = pd.DataFrame(all_note_rows)
all_tokens = [token for sequence in sequences for token in sequence]

print("\nParsed files with usable note sequences:", len(sequences))
print("Failed or skipped files:", len(failures) + (len(selected_midi_files) - len(sequences) - len(failures)))
print("Total note tokens:", len(all_tokens))

file_summary_df.head()

Processed  25/200 files


Processed  50/200 files


Processed  75/200 files


Processed 100/200 files


Processed 125/200 files


Processed 150/200 files


Processed 175/200 files


Processed 200/200 files



Parsed files with usable note sequences: 200
Failed or skipped files: 0
Total note tokens: 1006924


,file,note_tokens,first_pitch,last_pitch
0,maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1...,6493,71,55
1,maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1...,1159,43,88
2,maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1...,2081,38,77
3,maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1...,4741,55,67
4,maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_05_R1...,12750,32,71


## 3. Exploratory Analysis

These summary statistics and plots show the symbolic distribution the model will learn from: pitches, quantized durations, and velocities. The Markov model only uses pitch and duration tokens; velocity is included as optional descriptive analysis.

In [5]:
if note_df.empty:
    raise RuntimeError("No notes were extracted. Check DATASET_DIR and MIDI parsing settings.")

sequence_lengths = np.array([len(sequence) for sequence in sequences], dtype=int)

eda_stats = {
    "midi_files_found": len(midi_files),
    "midi_files_selected": len(selected_midi_files),
    "usable_files": len(sequences),
    "failed_files": len(failures),
    "total_note_tokens": len(all_tokens),
    "average_notes_per_song": float(sequence_lengths.mean()),
    "median_notes_per_song": float(np.median(sequence_lengths)),
    "min_notes_per_song": int(sequence_lengths.min()),
    "max_notes_per_song": int(sequence_lengths.max()),
}

pd.DataFrame([eda_stats]).T.rename(columns={0: "value"})

,value
midi_files_found,1276.00
midi_files_selected,200.00
usable_files,200.00
failed_files,0.00
total_note_tokens,1006924.00
average_notes_per_song,5034.62
median_notes_per_song,4738.00
min_notes_per_song,712.00
max_notes_per_song,16005.00


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(note_df["pitch"], bins=np.arange(21, 109), color="#3568a8", alpha=0.9)
axes[0].set_title("Pitch distribution")
axes[0].set_xlabel("MIDI pitch")
axes[0].set_ylabel("Count")

duration_counts = note_df["duration_bucket"].value_counts().sort_index()
axes[1].bar([str(x) for x in duration_counts.index], duration_counts.values, color="#7a9a01", alpha=0.9)
axes[1].set_title("Quantized duration distribution")
axes[1].set_xlabel("Duration bucket (seconds)")
axes[1].set_ylabel("Count")

axes[2].hist(note_df["velocity"], bins=20, color="#b15f2a", alpha=0.9)
axes[2].set_title("Velocity distribution")
axes[2].set_xlabel("Velocity")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

/var/folders/q7/k2hy8_x17g94zrhjg82fykx40000gn/T/ipykernel_41938/3015359746.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
def plot_piano_roll(note_rows, title, max_seconds=30.0):
    """Draw a compact piano-roll view for a list of note metadata rows."""
    rows = [row for row in note_rows if row["start"] <= max_seconds]
    fig, ax = plt.subplots(figsize=(14, 4))
    for row in rows:
        ax.plot(
            [row["start"], row["end"]],
            [row["pitch"], row["pitch"]],
            color="#2f6f9f",
            linewidth=2,
            alpha=0.8,
        )
    ax.set_title(title)
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("MIDI pitch")
    ax.set_xlim(0, max_seconds)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(file_summary_df["note_tokens"], bins=30, color="#545c6a", alpha=0.9)
axes[0].set_title("Notes per MIDI file")
axes[0].set_xlabel("Extracted note tokens")
axes[0].set_ylabel("File count")

most_common_tokens = global_token_counts.most_common(20) if "global_token_counts" in globals() else Counter(all_tokens).most_common(20)
token_labels = [str(token) for token, _ in most_common_tokens]
token_values = [count for _, count in most_common_tokens]
axes[1].barh(token_labels[::-1], token_values[::-1], color="#8d4f9f", alpha=0.9)
axes[1].set_title("Top token frequencies")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

example_notes = [row for row in all_note_rows if row["file"] == file_summaries[0]["file"]]
plot_piano_roll(example_notes, "Training example piano roll (first 30 seconds)", max_seconds=30.0)

/var/folders/q7/k2hy8_x17g94zrhjg82fykx40000gn/T/ipykernel_41938/1704336410.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/q7/k2hy8_x17g94zrhjg82fykx40000gn/T/ipykernel_41938/1704336410.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Model Formulation

The task is autoregressive sequence modeling. Given symbolic tokens `t_1, ..., t_n`, the model predicts the next token `t_{n+1}`. For a neural model, this corresponds to minimizing cross-entropy loss over the next-token distribution; perplexity is `exp(cross_entropy)`.

Models considered:

- **Random baseline:** sample tokens independently from the global token distribution. This is the simplest sanity check but produces incoherent music because it ignores context.
- **Trigram Markov chain:** estimate `P(t_n | t_{n-2}, t_{n-1})` from transition counts. It is fast and interpretable, but has weak long-range structure and can become repetitive.
- **LSTM language model:** use embeddings plus recurrent hidden state to model longer temporal dependencies. It is effective for sequence tasks, but training is sequential and it can still struggle with very long contexts.
- **Transformer decoder:** use self-attention and causal masking to model long-range structure with parallel training. It is powerful but more computationally expensive and memory intensive.

This notebook trains the random baseline, Markov baseline, and a compact LSTM. A small Transformer decoder class is included to show the natural extension, but it is not trained by default so the notebook remains lightweight.

## 5. Markov Baseline Training and Vocabulary Construction

The Markov training loop collects transition counts for every piece independently, so contexts do not cross file boundaries. The same extracted tokens are also converted into a vocabulary for the neural model: each unique `(pitch, duration_bucket)` token receives an integer ID.

In [8]:
def train_ngram_model(sequences, n=3):
    """Train an n-gram model represented by context -> Counter(next_token)."""
    if n < 2:
        raise ValueError("n must be at least 2")

    transitions = defaultdict(Counter)
    global_token_counts = Counter()

    context_size = n - 1
    for sequence in sequences:
        global_token_counts.update(sequence)
        for index in range(context_size, len(sequence)):
            context = tuple(sequence[index - context_size:index])
            target = sequence[index]
            transitions[context][target] += 1

    return dict(transitions), global_token_counts


def weighted_sample(counter, rng):
    """Sample one key from a Counter using counts as weights."""
    items = list(counter.items())
    choices = [item for item, _ in items]
    weights = [weight for _, weight in items]
    return rng.choices(choices, weights=weights, k=1)[0]


transition_model, global_token_counts = train_ngram_model(sequences, n=N_GRAM)
known_contexts = list(transition_model.keys())

# Vocabulary for neural autoregressive modeling.
vocab_tokens = sorted(global_token_counts.keys(), key=lambda token: (token[0], token[1]))
token_to_id = {token: index for index, token in enumerate(vocab_tokens)}
id_to_token = {index: token for token, index in token_to_id.items()}
sequence_ids = [[token_to_id[token] for token in sequence] for sequence in sequences]

print(f"n-gram order:        {N_GRAM}")
print(f"Unique contexts:     {len(known_contexts)}")
print(f"Unique tokens/vocab: {len(global_token_counts)}")
print(f"Total transitions:   {sum(sum(counter.values()) for counter in transition_model.values())}")

n-gram order:        3
Unique contexts:     69222
Unique tokens/vocab: 498
Total transitions:   1006524


## 6. Generation

Generation starts from a random context seen during training. At each step, the next token is sampled from the learned transition counts. If the current context is unseen, the sampler falls back to a random known context; if needed, it can also sample from the global token distribution.

In [9]:
def generate_markov_tokens(model, global_counts, length=300, n=3, rng=None):
    """Generate a token sequence from an n-gram Markov model."""
    if rng is None:
        rng = random.Random()
    if not model:
        raise ValueError("The Markov model is empty.")

    context_size = n - 1
    known_contexts = list(model.keys())
    current_context = list(rng.choice(known_contexts))
    generated = current_context.copy()

    while len(generated) < length:
        context = tuple(generated[-context_size:])
        transition_counts = model.get(context)

        if not transition_counts:
            context = rng.choice(known_contexts)
            transition_counts = model.get(context)

        if transition_counts:
            next_token = weighted_sample(transition_counts, rng)
        else:
            next_token = weighted_sample(global_counts, rng)

        generated.append(next_token)

    return generated[:length]


def generate_random_baseline_tokens(global_counts, length=300, rng=None):
    """Generate independent tokens from the global token distribution."""
    if rng is None:
        rng = random.Random()
    return [weighted_sample(global_counts, rng) for _ in range(length)]


markov_tokens = generate_markov_tokens(
    transition_model,
    global_token_counts,
    length=GENERATED_NOTE_COUNT,
    n=N_GRAM,
    rng=rng,
)

baseline_rng = random.Random(RANDOM_SEED + 1)
baseline_tokens = generate_random_baseline_tokens(
    global_token_counts,
    length=GENERATED_NOTE_COUNT,
    rng=baseline_rng,
)

print("First 12 Markov tokens:")
print(markov_tokens[:12])
print("\nFirst 12 random-baseline tokens:")
print(baseline_tokens[:12])

First 12 Markov tokens:
[(33, 0.25), (60, 0.125), (63, 0.25), (72, 0.125), (69, 0.25), (76, 0.125), (71, 0.125), (70, 0.125), (71, 0.125), (72, 0.125), (67, 0.125), (64, 0.125)]

First 12 random-baseline tokens:
[(62, 0.125), (45, 0.125), (74, 1.0), (73, 0.125), (60, 0.5), (68, 0.25), (84, 0.25), (77, 0.125), (71, 0.5), (81, 0.125), (78, 0.125), (39, 0.25)]


In [10]:
def sequence_appears_as_training_subsequence(generated_tokens, training_sequences):
    """Check whether the full generated sequence appears contiguously in training data."""
    generated_tuple = tuple(generated_tokens)
    generated_length = len(generated_tuple)

    for sequence in training_sequences:
        if len(sequence) < generated_length:
            continue
        sequence_tuple = tuple(sequence)
        for start in range(0, len(sequence_tuple) - generated_length + 1):
            if sequence_tuple[start:start + generated_length] == generated_tuple:
                return True
    return False


def adapt_tokens_for_guitar(tokens, low_pitch=40, high_pitch=88):
    """Transpose notes by octaves into a practical guitar-like MIDI pitch range."""
    adapted = []
    for pitch, duration in tokens:
        new_pitch = int(pitch)
        while new_pitch < low_pitch:
            new_pitch += 12
        while new_pitch > high_pitch:
            new_pitch -= 12
        adapted.append((new_pitch, duration))
    return adapted


def tokens_to_midi(tokens, output_path, tempo_bpm=120, velocity=80, program=0, instrument_name="Generated Instrument"):
    """Convert note tokens back to a single-track MIDI file."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    midi = pretty_midi.PrettyMIDI(initial_tempo=tempo_bpm)
    instrument = pretty_midi.Instrument(program=program, is_drum=False, name=instrument_name)

    current_time = 0.0
    for pitch, duration in tokens:
        start = current_time
        end = start + float(duration)
        instrument.notes.append(
            pretty_midi.Note(
                velocity=int(velocity),
                pitch=int(pitch),
                start=float(start),
                end=float(end),
            )
        )
        current_time = end

    midi.instruments.append(instrument)
    midi.write(str(output_path))
    return output_path


def optionally_render_with_fluidsynth(midi_path, wav_path, sf2_path=None):
    """Optional audio rendering helper. Requires an installed FluidSynth binary and a local .sf2 file."""
    if sf2_path is None or not Path(sf2_path).exists() or shutil.which("fluidsynth") is None:
        return None
    command = ["fluidsynth", "-ni", str(sf2_path), str(midi_path), "-F", str(wav_path), "-r", "44100"]
    subprocess.run(command, check=True)
    return Path(wav_path)


markov_guitar_tokens = adapt_tokens_for_guitar(markov_tokens)
baseline_guitar_tokens = adapt_tokens_for_guitar(baseline_tokens)
is_training_copy = sequence_appears_as_training_subsequence(markov_tokens, sequences)
print("Generated Markov sequence appears as exact training subsequence:", is_training_copy)

markov_path = tokens_to_midi(
    markov_guitar_tokens,
    MARKOV_OUTPUT_PATH,
    tempo_bpm=TEMPO_BPM,
    velocity=GENERATED_VELOCITY,
    program=GM_PROGRAMS["acoustic_guitar_steel"],
    instrument_name="Task 1 Markov Acoustic Guitar",
)
baseline_path = tokens_to_midi(
    baseline_guitar_tokens,
    BASELINE_OUTPUT_PATH,
    tempo_bpm=TEMPO_BPM,
    velocity=GENERATED_VELOCITY,
    program=GM_PROGRAMS["acoustic_guitar_steel"],
    instrument_name="Task 1 Random Baseline Acoustic Guitar",
)

print(f"Saved Markov acoustic-guitar MIDI: {markov_path.resolve()}")
print(f"Saved random baseline MIDI:        {baseline_path.resolve()}")

Generated Markov sequence appears as exact training subsequence: False
Saved Markov acoustic-guitar MIDI: /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1/markov_acoustic_guitar.mid
Saved random baseline MIDI:        /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1/random_baseline.mid


## 7. Neural Autoregressive Model: LSTM and Transformer Decoder

The LSTM uses the constructed token vocabulary. For each fixed-length window, the input is a sequence of token IDs and the target is the same sequence shifted one step forward. Cross-entropy loss trains the model to assign high probability to the next observed token. The validation split is by MIDI file rather than by random note, which makes the perplexity check slightly more honest.

In [11]:
if not TORCH_AVAILABLE:
    raise RuntimeError(f"PyTorch is required for this section but could not be imported: {TORCH_IMPORT_ERROR}")


class TokenWindowDataset(Dataset):
    """Fixed-length next-token windows from token-ID sequences."""

    def __init__(self, id_sequences, context_length=64, max_windows=None, seed=42):
        self.id_sequences = id_sequences
        self.context_length = context_length
        self.index = []
        for seq_idx, ids in enumerate(id_sequences):
            max_start = len(ids) - context_length - 1
            if max_start <= 0:
                continue
            # Strided windows avoid creating millions of nearly identical examples.
            step = max(1, context_length // 2)
            for start in range(0, max_start, step):
                self.index.append((seq_idx, start))
        local_rng = random.Random(seed)
        local_rng.shuffle(self.index)
        if max_windows is not None:
            self.index = self.index[:max_windows]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, item):
        seq_idx, start = self.index[item]
        ids = self.id_sequences[seq_idx]
        x = torch.tensor(ids[start:start + self.context_length], dtype=torch.long)
        y = torch.tensor(ids[start + 1:start + self.context_length + 1], dtype=torch.long)
        return x, y


class LSTMMusicModel(nn.Module):
    """Compact token-level LSTM language model."""

    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=2, dropout=0.15):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, hidden=None):
        embedded = self.embedding(token_ids)
        output, hidden = self.lstm(embedded, hidden)
        logits = self.output(output)
        return logits, hidden


class TransformerDecoderMusicModel(nn.Module):
    """Small causal Transformer decoder-style language model for comparison."""

    def __init__(self, vocab_size, context_length=64, embedding_dim=128, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=n_heads,
            dim_feedforward=embedding_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=token_ids.device), diagonal=1).bool()
        x = self.transformer(x, mask=causal_mask)
        return self.output(x)


split_index = max(1, int(0.8 * len(sequence_ids)))
train_id_sequences = sequence_ids[:split_index]
val_id_sequences = sequence_ids[split_index:]

train_dataset = TokenWindowDataset(
    train_id_sequences,
    context_length=CONTEXT_LENGTH,
    max_windows=MAX_TRAIN_WINDOWS,
    seed=RANDOM_SEED,
)
val_dataset = TokenWindowDataset(
    val_id_sequences,
    context_length=CONTEXT_LENGTH,
    max_windows=MAX_VAL_WINDOWS,
    seed=RANDOM_SEED + 1,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Vocabulary size:", len(token_to_id))
print("Train files:", len(train_id_sequences), "Validation files:", len(val_id_sequences))
print("Train windows:", len(train_dataset), "Validation windows:", len(val_dataset))

Vocabulary size: 498
Train files: 160 Validation files: 40
Train windows: 5000 Validation windows: 1000


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMMusicModel(
    vocab_size=len(token_to_id),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def run_epoch(model, data_loader, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_tokens = 0

    for x, y in data_loader:
        x = x.to(device)
        y = y.to(device)
        if is_training:
            optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        if is_training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        token_count = y.numel()
        total_loss += float(loss.item()) * token_count
        total_tokens += token_count

    average_loss = total_loss / max(1, total_tokens)
    return average_loss, math.exp(min(average_loss, 20))


history = []
for epoch in range(1, LSTM_EPOCHS + 1):
    train_loss, train_ppl = run_epoch(model, train_loader, optimizer=optimizer)
    with torch.no_grad():
        val_loss, val_ppl = run_epoch(model, val_loader, optimizer=None)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_perplexity": train_ppl,
            "val_loss": val_loss,
            "val_perplexity": val_ppl,
        }
    )
    print(
        f"Epoch {epoch}: "
        f"train loss={train_loss:.3f}, train ppl={train_ppl:.1f}, "
        f"val loss={val_loss:.3f}, val ppl={val_ppl:.1f}"
    )

history_df = pd.DataFrame(history)
history_df

Epoch 1: train loss=5.298, train ppl=200.0, val loss=5.154, val ppl=173.1


Epoch 2: train loss=4.983, train ppl=146.0, val loss=4.999, val ppl=148.2


Epoch 3: train loss=4.858, train ppl=128.8, val loss=4.884, val ppl=132.2


,epoch,train_loss,train_perplexity,val_loss,val_perplexity
0,1,5.298452,200.026939,5.154055,173.132109
1,2,4.983304,145.955775,4.998881,148.247205
2,3,4.857969,128.762402,4.884109,132.172684


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
axes[0].set_title("Cross-entropy loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_perplexity"], marker="o", label="train")
axes[1].plot(history_df["epoch"], history_df["val_perplexity"], marker="o", label="validation")
axes[1].set_title("Perplexity")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Perplexity")
axes[1].legend()

plt.tight_layout()
plt.show()

/var/folders/q7/k2hy8_x17g94zrhjg82fykx40000gn/T/ipykernel_41938/2922756953.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
def sample_from_logits(logits, temperature=1.0, top_k=None):
    """Temperature and top-k sampling from one logits vector."""
    logits = logits.detach().float().cpu()
    if temperature <= 0:
        return int(torch.argmax(logits).item())
    logits = logits / temperature
    if top_k is not None and top_k < logits.numel():
        values, indices = torch.topk(logits, k=top_k)
        probs = torch.softmax(values, dim=-1)
        sampled_position = torch.multinomial(probs, num_samples=1).item()
        return int(indices[sampled_position].item())
    probs = torch.softmax(logits, dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())


def generate_lstm_tokens(model, seed_tokens, length=300, context_length=64, temperature=1.20, top_k=40):
    """Generate new tokens from an LSTM, using seed_tokens only as context."""
    model.eval()
    context_ids = [token_to_id[token] for token in seed_tokens]
    generated_ids = []
    with torch.no_grad():
        while len(generated_ids) < length:
            x_ids = context_ids[-context_length:]
            x = torch.tensor(x_ids, dtype=torch.long, device=device).unsqueeze(0)
            logits, _ = model(x)
            next_id = sample_from_logits(logits[0, -1], temperature=temperature, top_k=top_k)
            generated_ids.append(next_id)
            context_ids.append(next_id)
    return [id_to_token[index] for index in generated_ids]


seed_sequence = rng.choice(sequences)
seed_start = rng.randint(0, max(0, len(seed_sequence) - CONTEXT_LENGTH - 1))
seed_tokens = seed_sequence[seed_start:seed_start + CONTEXT_LENGTH]

lstm_tokens = generate_lstm_tokens(
    model,
    seed_tokens=seed_tokens,
    length=GENERATED_NOTE_COUNT,
    context_length=CONTEXT_LENGTH,
    temperature=TEMPERATURE,
    top_k=TOP_K,
)
lstm_guitar_tokens = adapt_tokens_for_guitar(lstm_tokens)

lstm_is_training_copy = sequence_appears_as_training_subsequence(lstm_tokens, sequences)
print("Generated LSTM sequence appears as exact training subsequence:", lstm_is_training_copy)
print("First 12 LSTM tokens:", lstm_tokens[:12])

# Main required output: symbolic, unconditioned, acoustic-guitar MIDI.
final_path = tokens_to_midi(
    lstm_guitar_tokens,
    FINAL_OUTPUT_PATH,
    tempo_bpm=TEMPO_BPM,
    velocity=GENERATED_VELOCITY,
    program=GM_PROGRAMS["acoustic_guitar_steel"],
    instrument_name="Task 1 LSTM Acoustic Guitar",
)

# Additional MIDI variants for piano/guitar comparison.
lstm_piano_path = tokens_to_midi(
    lstm_tokens,
    LSTM_PIANO_OUTPUT_PATH,
    tempo_bpm=TEMPO_BPM,
    velocity=GENERATED_VELOCITY,
    program=GM_PROGRAMS["piano"],
    instrument_name="Task 1 LSTM Piano",
)
lstm_acoustic_path = tokens_to_midi(
    lstm_guitar_tokens,
    LSTM_ACOUSTIC_GUITAR_OUTPUT_PATH,
    tempo_bpm=TEMPO_BPM,
    velocity=GENERATED_VELOCITY,
    program=GM_PROGRAMS["acoustic_guitar_steel"],
    instrument_name="Task 1 LSTM Acoustic Guitar",
)
lstm_electric_path = tokens_to_midi(
    lstm_guitar_tokens,
    LSTM_ELECTRIC_GUITAR_OUTPUT_PATH,
    tempo_bpm=TEMPO_BPM,
    velocity=GENERATED_VELOCITY,
    program=GM_PROGRAMS["electric_guitar_clean"],
    instrument_name="Task 1 LSTM Electric Guitar",
)

print(f"Saved required final MIDI:      {final_path.resolve()}")
print(f"Saved LSTM piano comparison:    {lstm_piano_path.resolve()}")
print(f"Saved LSTM acoustic comparison: {lstm_acoustic_path.resolve()}")
print(f"Saved LSTM electric comparison: {lstm_electric_path.resolve()}")

Generated LSTM sequence appears as exact training subsequence: False
First 12 LSTM tokens: [(52, 0.125), (73, 0.125), (43, 0.125), (62, 0.125), (72, 0.125), (72, 0.125), (79, 0.125), (73, 0.125), (79, 0.125), (85, 0.125), (62, 0.125), (66, 0.125)]
Saved required final MIDI:      /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1/symbolic_unconditioned.mid
Saved LSTM piano comparison:    /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1/lstm_piano.mid
Saved LSTM acoustic comparison: /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1/lstm_acoustic_guitar.mid
Saved LSTM electric comparison: /Users/chark1es/Stuff/GitHub/CSE153-253-Assignment2/Task1/lstm_electric_guitar.mid


In [15]:
def tokens_to_note_rows(tokens):
    rows = []
    current_time = 0.0
    for pitch, duration in tokens:
        rows.append({"start": current_time, "end": current_time + float(duration), "pitch": int(pitch)})
        current_time += float(duration)
    return rows


plot_piano_roll(tokens_to_note_rows(lstm_guitar_tokens), "Generated LSTM acoustic-guitar piano roll", max_seconds=30.0)
plot_piano_roll(tokens_to_note_rows(markov_guitar_tokens), "Generated Markov acoustic-guitar piano roll", max_seconds=30.0)

/var/folders/q7/k2hy8_x17g94zrhjg82fykx40000gn/T/ipykernel_41938/1704336410.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Evaluation

The evaluation compares the training distribution with random, Markov, and LSTM generated outputs. The random baseline samples notes independently from the global token distribution, while the Markov and LSTM models use local context. These metrics do not measure musical quality directly, but they help confirm whether the generated symbolic distributions are plausible relative to the training data.

A good symbolic output should use a plausible pitch range, maintain rhythmic consistency, avoid degenerate repetition, avoid unrealistic note density, and sound more locally coherent than independent random notes. Cross-entropy/perplexity measures next-token prediction quality, while listening tests and piano-roll inspection are still needed to judge subjective musical properties and whether piano-trained patterns sound natural when rendered as guitar.

In [16]:
def summarize_token_sequence(name, tokens):
    pitches = np.array([pitch for pitch, _ in tokens], dtype=int)
    durations = np.array([duration for _, duration in tokens], dtype=float)
    repeated_bigrams = sum(
        1 for i in range(2, len(tokens))
        if tokens[i - 2:i] == tokens[i - 1:i + 1]
    )

    return {
        "name": name,
        "note_count": int(len(tokens)),
        "average_pitch": float(pitches.mean()),
        "min_pitch": int(pitches.min()),
        "max_pitch": int(pitches.max()),
        "pitch_range": int(pitches.max() - pitches.min()),
        "unique_pitch_count": int(len(set(pitches.tolist()))),
        "average_duration": float(durations.mean()),
        "repeated_bigram_rate": repeated_bigrams / max(1, len(tokens) - 2),
    }


summary_df = pd.DataFrame(
    [
        summarize_token_sequence("training", all_tokens),
        summarize_token_sequence("random_baseline", baseline_tokens),
        summarize_token_sequence("markov", markov_tokens),
        summarize_token_sequence("lstm", lstm_tokens),
        summarize_token_sequence("lstm_guitar_adapted", lstm_guitar_tokens),
    ]
)

summary_df

,name,note_count,average_pitch,min_pitch,max_pitch,pitch_range,unique_pitch_count,average_duration,repeated_bigram_rate
0,training,1006924,63.959955,21,108,87,88,0.263629,0.002194
1,random_baseline,300,63.463333,26,101,75,63,0.280417,0.000000
2,markov,300,66.546667,33,102,69,61,0.238750,0.000000
3,lstm,300,70.136667,43,90,47,45,0.125000,0.000000
4,lstm_guitar_adapted,300,70.096667,43,88,45,44,0.125000,0.000000


In [17]:
def categorical_distribution(values, categories, smoothing=1e-9):
    counts = Counter(values)
    probs = np.array([counts.get(category, 0) for category in categories], dtype=float)
    probs = probs + smoothing
    return probs / probs.sum()


def l1_distance(p, q):
    return float(np.sum(np.abs(p - q)))


def kl_divergence(p, q):
    return float(np.sum(p * np.log(p / q)))


pitch_categories = list(range(128))
duration_categories = [float(x) for x in DURATION_BUCKETS]

generated_sets = {
    "random_baseline": baseline_tokens,
    "markov": markov_tokens,
    "lstm": lstm_tokens,
}

training_pitches = [pitch for pitch, _ in all_tokens]
training_durations = [duration for _, duration in all_tokens]
train_pitch_dist = categorical_distribution(training_pitches, pitch_categories)
train_duration_dist = categorical_distribution(training_durations, duration_categories)

rows = []
distributions = {"training": (train_pitch_dist, train_duration_dist)}
for name, tokens in generated_sets.items():
    pitches = [pitch for pitch, _ in tokens]
    durations = [duration for _, duration in tokens]
    pitch_dist = categorical_distribution(pitches, pitch_categories)
    duration_dist = categorical_distribution(durations, duration_categories)
    distributions[name] = (pitch_dist, duration_dist)
    rows.append(
        {
            "model": name,
            "pitch_L1_vs_training": l1_distance(train_pitch_dist, pitch_dist),
            "duration_L1_vs_training": l1_distance(train_duration_dist, duration_dist),
            "pitch_KL_training_to_model": kl_divergence(train_pitch_dist, pitch_dist),
            "duration_KL_training_to_model": kl_divergence(train_duration_dist, duration_dist),
        }
    )

distance_df = pd.DataFrame(rows)
distance_df

,model,pitch_L1_vs_training,duration_L1_vs_training,pitch_KL_training_to_model,duration_KL_training_to_model
0,random_baseline,0.355079,0.086235,1.062384,0.008899
1,markov,0.386662,0.100554,0.795344,0.014764
2,lstm,0.613363,0.698515,2.640210,8.187235


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

pitch_axis = np.array(pitch_categories)
for name, (pitch_dist, _) in distributions.items():
    linewidth = 2.5 if name == "training" else 1.8
    axes[0].plot(pitch_axis, pitch_dist, label=name, linewidth=linewidth)
axes[0].set_xlim(20, 110)
axes[0].set_title("Pitch distribution comparison")
axes[0].set_xlabel("MIDI pitch")
axes[0].set_ylabel("Probability")
axes[0].legend()

x = np.arange(len(duration_categories))
width = 0.2
for offset, (name, (_, duration_dist)) in enumerate(distributions.items()):
    axes[1].bar(x + (offset - 1.5) * width, duration_dist, width=width, label=name)
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(value) for value in duration_categories])
axes[1].set_title("Duration distribution comparison")
axes[1].set_xlabel("Duration bucket (seconds)")
axes[1].set_ylabel("Probability")
axes[1].legend()

plt.tight_layout()
plt.show()

/var/folders/q7/k2hy8_x17g94zrhjg82fykx40000gn/T/ipykernel_41938/660880107.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Related Work and Limitations

MAESTRO is widely used for symbolic music generation, automatic music transcription, expressive performance modeling, and neural synthesis research. Its main strength is the clean alignment of high-quality piano audio and MIDI performance data, including expressive timing and velocity. Similar symbolic generation work often uses event-based vocabularies with note-on, note-off, velocity, and time-shift tokens, then trains recurrent networks or Transformer language models.

Compared with that related work, this notebook is deliberately smaller. The LSTM and Markov models learn from the MAESTRO MIDI files directly, but the token representation is simpler than modern event-token systems. The generated guitar outputs demonstrate symbolic transfer to guitar timbres, but they may still contain piano-specific patterns such as large leaps, dense chord-like passages, or ranges that need octave adaptation for guitar.

Limitations:

- The note token includes pitch and duration bucket but not explicit harmony, bar position, key, sustain pedal, or expressive velocity control.
- Sequential MIDI reconstruction removes simultaneous chord timing. The piano-roll plots help inspect this, but a richer representation with time-shift/rest tokens would preserve polyphony better.
- The Markov chain captures only local transitions. The LSTM captures longer context but is still trained briefly for notebook runtime.
- The Transformer class shows a stronger modeling option, but full Transformer training would require more compute, memory, and tuning.
- General MIDI guitar programs provide a practical symbolic guitar output. High-quality audio rendering would require a local soundfont and synthesizer such as FluidSynth.